# 03 — Modeling & Evaluation
Train 6 classifiers, compare by ROC-AUC, evaluate the best.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')
from src.features import clean_data, create_features, encode_features
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import *

## Load & Prepare Data

In [ ]:
df = pd.read_csv('../data/telco-customer-churn.csv')
df = clean_data(df)
df = create_features(df)
df = encode_features(df)
print(f'Shape: {df.shape}, Features: {df.shape[1]-1}')

In [ ]:
X = df.drop(columns=['Churn'])
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')

## Train & Compare 6 Models

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), True),
    'Decision Tree': (DecisionTreeClassifier(random_state=42), False),
    'Random Forest': (RandomForestClassifier(n_estimators=100, random_state=42), False),
    'SVM': (SVC(probability=True, random_state=42), True),
    'KNN': (KNeighborsClassifier(), True),
    'Naive Bayes': (GaussianNB(), False),
}

results = []
for name, (m, sc) in models.items():
    Xtr = X_train_sc if sc else X_train.values
    Xte = X_test_sc if sc else X_test.values
    m.fit(Xtr, y_train)
    yp = m.predict(Xte)
    ypr = m.predict_proba(Xte)[:,1]
    results.append({'Model': name,
        'Accuracy': round(accuracy_score(y_test, yp), 4),
        'Precision': round(precision_score(y_test, yp), 4),
        'Recall': round(recall_score(y_test, yp), 4),
        'F1': round(f1_score(y_test, yp), 4),
        'ROC-AUC': round(roc_auc_score(y_test, ypr), 4)})

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
results_df

## Best Model Evaluation: Logistic Regression

In [ ]:
best = models['Logistic Regression'][0]
y_pred = best.predict(X_test_sc)
y_proba = best.predict_proba(X_test_sc)[:,1]
print(classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
print(f'Confusion matrix sum: {cm.sum()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No','Yes'], yticklabels=['No','Yes'])
axes[0].set_title('Confusion Matrix', fontweight='bold', color='#0F2040')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='#0E9F8A', lw=2, label=f'AUC = {roc_auc_score(y_test, y_proba):.3f}')
axes[1].plot([0,1],[0,1],'--', color='#999')
axes[1].set_title('ROC Curve', fontweight='bold', color='#0F2040')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()
plt.tight_layout()
plt.show()

## Feature Importance

In [ ]:
# Logistic Regression coefficients
lr_imp = pd.Series(np.abs(best.coef_[0]), index=X.columns).sort_values(ascending=True).tail(10)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
lr_imp.plot(kind='barh', color='#0E9F8A', ax=axes[0])
axes[0].set_title('LR Feature Importance (|coef|)', fontweight='bold', color='#0F2040')
# Random Forest
rf = models['Random Forest'][0]
rf_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True).tail(10)
rf_imp.plot(kind='barh', color='#0F2040', ax=axes[1])
axes[1].set_title('RF Feature Importance', fontweight='bold', color='#0F2040')
plt.tight_layout()
plt.show()
print('Top features: TotalCharges, tenure, AvgCharge — as expected.')
print('\n⚖️ Trade-off: Recall matters more than accuracy here.')
print('A missed churner costs more (lost revenue) than a false alarm (unnecessary retention offer).')